# Metaflora Incubus v1: Bonsai advisory

This is a separate private Kaggle benchmark. It never changes the mandatory promotion head-to-head run. The run only begins when enough local disk remains for the advisory artifact plus a 1 GiB safety reserve; otherwise it writes a private skip receipt and exits successfully. Nothing is published.

In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

WORK_ROOT = Path('/kaggle/working')
REPO = WORK_ROOT / 'metaflora-incubus-bonsai-advisory'
MODEL_DIR = WORK_ROOT / 'bonsai-advisory-models'
OUTPUT_DIR = WORK_ROOT / 'bonsai-advisory-output'
MANIFEST_PATH = WORK_ROOT / 'bonsai-advisory-v1.json'
SKIP_PATH = OUTPUT_DIR / 'bonsai-advisory-skip.json'
if not WORK_ROOT.is_dir():
    raise RuntimeError('This notebook must run in a Kaggle session')
code_revision = "7ec9bcd46001b0ecd8d15e83203835f06dca59ea"
if re.fullmatch(r'[0-9a-f]{40}', code_revision) is None:
    raise RuntimeError('INCUBUS_CODE_REVISION must be an exact lowercase commit SHA')
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(['git', 'init', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'remote', 'add', 'origin', 'https://github.com/metaflora-app/metaflora-incubus.git'], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', code_revision], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
checked_out = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip().lower()
if checked_out != code_revision:
    raise RuntimeError('trusted code revision verification failed')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--require-hashes', '-r', str(REPO / 'requirements/h2h-linux.lock')], check=True)
sys.path.insert(0, str(REPO / 'src'))
from huggingface_hub import HfApi, hf_hub_download
from metaflora_incubus.cloud_bootstrap import decrypt_cloud_bootstrap, install_cloud_bootstrap

encoded_bootstrap = Path('/kaggle/input/incubus-private-runtime-bootstrap/bootstrap-key.txt').read_text(encoding='utf-8').strip()
if not encoded_bootstrap:
    raise RuntimeError('INCUBUS_BOOTSTRAP is required for private checkpoint access')
bootstrap_payload = decrypt_cloud_bootstrap((REPO / 'configs/cloud/bootstrap-v1.enc').read_bytes(), encoded_bootstrap)
install_cloud_bootstrap(bootstrap_payload, home=Path.home(), environment=os.environ)
checkpoint_repo = os.environ.get('INCUBUS_CHECKPOINT_LOCATION', 'metaflora/incubus-checkpoints')
checkpoint_branch = os.environ.get('INCUBUS_CHECKPOINT_BRANCH', 'incubus-training-v1')
checkpoint_token = os.environ.get('HF_TOKEN') or None
checkpoint_head = HfApi(token=checkpoint_token).model_info(repo_id=checkpoint_repo, revision=checkpoint_branch).sha
if not isinstance(checkpoint_head, str) or re.fullmatch(r'[0-9a-f]{40}', checkpoint_head) is None:
    raise RuntimeError('private checkpoint revision could not be frozen')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
private_cache = MODEL_DIR / 'private-checkpoint-cache'
print({'code_revision': checked_out, 'checkpoint_revision': checkpoint_head, 'output': str(OUTPUT_DIR)})


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def atomic_json(path, document):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(document, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    temporary.replace(path)

def verify_gguf(path, expected_size, expected_sha256):
    path = Path(path)
    with path.open('rb') as stream:
        if stream.read(4) != b'GGUF':
            raise RuntimeError(f'Invalid GGUF magic: {path}')
    if path.stat().st_size != expected_size:
        raise RuntimeError(f'GGUF size mismatch: {path}')
    digest = sha256_file(path)
    if digest != expected_sha256:
        raise RuntimeError(f'GGUF SHA-256 mismatch: {path}')
    return {'path': str(path), 'size': path.stat().st_size, 'sha256': digest}

def download_private(name, revision):
    return Path(hf_hub_download(repo_id=checkpoint_repo, filename=name, revision=revision, cache_dir=private_cache, token=checkpoint_token, force_download=False))

candidate_pointer_name = 'runs/incubus-v1-refine-001/exports/candidate-upload-receipt.json'
candidate_upload_receipt = json.loads(download_private(candidate_pointer_name, checkpoint_head).read_text(encoding='utf-8'))
candidate_artifact_revision = candidate_upload_receipt['artifact_revision']
candidate_artifact_path = candidate_upload_receipt['artifact_path']
candidate_remote_prefix = candidate_upload_receipt['remote_prefix']
candidate_artifact_sha256 = candidate_upload_receipt['artifact_sha256']
candidate_artifact_size_bytes = candidate_upload_receipt['artifact_size_bytes']
if not isinstance(candidate_artifact_revision, str) or re.fullmatch(r'[0-9a-f]{40}', candidate_artifact_revision) is None:
    raise RuntimeError('candidate artifact revision is invalid')
if not isinstance(candidate_artifact_sha256, str) or re.fullmatch(r'[0-9a-f]{64}', candidate_artifact_sha256) is None:
    raise RuntimeError('candidate artifact digest is invalid')
if not isinstance(candidate_artifact_size_bytes, int) or candidate_artifact_size_bytes <= 0:
    raise RuntimeError('candidate artifact size is invalid')
CANDIDATE_GGUF = download_private(candidate_artifact_path, candidate_artifact_revision)
candidate_export = json.loads(download_private(f'{candidate_remote_prefix}/candidate-export.json', candidate_artifact_revision).read_text(encoding='utf-8'))
if candidate_export.get('artifact', {}).get('sha256') != candidate_artifact_sha256:
    raise RuntimeError('candidate export receipt binding failed')
candidate_verification = verify_gguf(CANDIDATE_GGUF, candidate_artifact_size_bytes, candidate_artifact_sha256)

incumbent_name = 'runs/incubus-v1-run/artifacts/metaflora-incubus-v1.gguf'
incumbent_receipt_name = 'runs/incubus-v1-run/artifacts/artifact-metadata.json'
INCUMBENT_GGUF = download_private(incumbent_name, checkpoint_head)
incumbent_receipt = json.loads(download_private(incumbent_receipt_name, checkpoint_head).read_text(encoding='utf-8'))
incumbent_verification = verify_gguf(INCUMBENT_GGUF, incumbent_receipt['artifact_size_bytes'], incumbent_receipt['artifact_sha256'])

llama_cpp = WORK_ROOT / 'llama.cpp-bonsai-advisory'
LLAMA_SERVER = llama_cpp / 'build/bin/llama-server'
LLAMA_SERVER_BUILD_LOG = OUTPUT_DIR / 'llama-server-build.log'
llama_cpp_revision = 'bf2c86ddc0685f580595954056c2e77ebabfab4f'
if re.fullmatch(r'[0-9a-f]{40}', llama_cpp_revision) is None:
    raise RuntimeError('llama.cpp revision is not pinned')

def run_build_step(label, command):
    completed = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    with LLAMA_SERVER_BUILD_LOG.open('a', encoding='utf-8') as stream:
        stream.write(f"$ {subprocess.list2cmdline([str(item) for item in command])}\n")
        stream.write(completed.stdout or '')
        stream.write('\n')
    return completed

def raise_build_error(label, completed, summary):
    tail = (completed.stdout or '').strip().splitlines()[-20:]
    detail = '\n'.join(tail) or 'no compiler output captured'
    raise RuntimeError(f"{summary} during {label}; log: {LLAMA_SERVER_BUILD_LOG}\n{detail}")

shutil.rmtree(llama_cpp, ignore_errors=True)
LLAMA_SERVER_BUILD_LOG.unlink(missing_ok=True)
for label, command in (
    ('fetch', ['git', 'init', str(llama_cpp)]),
    ('add-remote', ['git', '-C', str(llama_cpp), 'remote', 'add', 'origin', 'https://github.com/ggerganov/llama.cpp.git']),
    ('fetch', ['git', '-C', str(llama_cpp), 'fetch', '--depth', '1', 'origin', llama_cpp_revision]),
    ('checkout', ['git', '-C', str(llama_cpp), 'checkout', '--detach', 'FETCH_HEAD']),
):
    completed = run_build_step(label, command)
    if completed.returncode != 0:
        raise_build_error(label, completed, 'llama-server build failed')
checked_out_llama_revision = subprocess.check_output(['git', '-C', str(llama_cpp), 'rev-parse', 'HEAD'], text=True).strip().lower()
if checked_out_llama_revision != llama_cpp_revision:
    raise RuntimeError('llama.cpp checkout verification failed')
cuda_configure = run_build_step('cuda_configure', ['cmake', '-S', str(llama_cpp), '-B', str(llama_cpp / 'build'), '-DGGML_CUDA=ON', '-DLLAMA_CURL=OFF', '-DCMAKE_BUILD_TYPE=Release'])
cuda_compile = None
if cuda_configure.returncode == 0:
    cuda_compile = run_build_step('cuda_compile', ['cmake', '--build', str(llama_cpp / 'build'), '--config', 'Release', '-j', '2', '--target', 'llama-server'])
server_backend = 'cuda'
if cuda_configure.returncode != 0 or cuda_compile is None or cuda_compile.returncode != 0:
    server_backend = 'cpu_fallback'
    shutil.rmtree(llama_cpp / 'build', ignore_errors=True)
    cpu_configure = run_build_step('cpu_fallback_configure', ['cmake', '-S', str(llama_cpp), '-B', str(llama_cpp / 'build'), '-DGGML_CUDA=OFF', '-DLLAMA_CURL=OFF', '-DCMAKE_BUILD_TYPE=Release'])
    if cpu_configure.returncode != 0:
        raise_build_error('cpu_fallback_configure', cpu_configure, 'CPU fallback build failed')
    cpu_compile = run_build_step('cpu_fallback_compile', ['cmake', '--build', str(llama_cpp / 'build'), '--config', 'Release', '-j', '2', '--target', 'llama-server'])
    if cpu_compile.returncode != 0:
        raise_build_error('cpu_fallback_compile', cpu_compile, 'CPU fallback build failed')
if not LLAMA_SERVER.is_file():
    raise FileNotFoundError(f'llama-server build did not produce {LLAMA_SERVER}; inspect {LLAMA_SERVER_BUILD_LOG}')
LLAMA_SERVER.chmod(0o700)

catalog = json.loads((REPO / 'benchmarks/head-to-head-v1-baselines.json').read_text(encoding='utf-8'))
same_size_pin = next(pin for pin in catalog['baselines'] if pin['id'] == 'qwen3-4b-instruct-2507')
bonsai_pin = next(pin for pin in catalog['baselines'] if pin['id'] == 'bonsai-27b')
if bonsai_pin['role'] != 'conditional' or bonsai_pin['promotion_required']:
    raise RuntimeError('Bonsai must remain an advisory-only comparator')

def download_verified(pin):
    path = Path(hf_hub_download(repo_id=pin['artifact_repo_id'], filename=pin['artifact_filename'], revision=pin['artifact_revision'], local_dir=MODEL_DIR, token=os.environ.get('HF_TOKEN') or None, force_download=False))
    return verify_gguf(path, pin['artifact_size_bytes'], pin['artifact_sha256'])

same_size_verification = download_verified(same_size_pin)
same_size_path = Path(same_size_verification['path'])
expected_bonsai_size = 3_803_452_480
if bonsai_pin['artifact_size_bytes'] != expected_bonsai_size:
    raise RuntimeError('Bonsai catalog size changed; advisory run is refused')
minimum_free_bytes = 4_877_194_304
available_free_bytes = shutil.disk_usage(WORK_ROOT).free
if available_free_bytes < minimum_free_bytes:
    atomic_json(SKIP_PATH, {'schema_version': 1, 'status': 'skipped', 'reason': 'insufficient_disk', 'available_free_bytes': available_free_bytes, 'required_free_bytes': minimum_free_bytes, 'artifact_size_bytes': expected_bonsai_size, 'artifact_id': bonsai_pin['id']})
    print({'status': 'skipped', 'receipt': str(SKIP_PATH)})
    raise SystemExit(0)
try:
    bonsai_verification = download_verified(bonsai_pin)
except Exception as error:
    atomic_json(SKIP_PATH, {'schema_version': 1, 'status': 'skipped', 'reason': 'download_or_verification_failed', 'error_type': type(error).__name__, 'artifact_id': bonsai_pin['id'], 'required_free_bytes': minimum_free_bytes})
    print({'status': 'skipped', 'receipt': str(SKIP_PATH)})
    raise SystemExit(0)
bonsai_path = Path(bonsai_verification['path'])
print({'candidate': candidate_verification['path'], 'same_size': same_size_verification['path'], 'bonsai': bonsai_verification['path'], 'free_bytes_before_bonsai': available_free_bytes})


In [ ]:
runner_revision = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
launch_settings = catalog['launch_settings']
models = [
    {'id': 'candidate-v1', 'role': 'candidate', 'path': str(CANDIDATE_GGUF), 'sha256': candidate_verification['sha256']},
    {'id': 'incumbent-v1', 'role': 'incumbent', 'path': str(INCUMBENT_GGUF), 'sha256': incumbent_verification['sha256']},
    {'id': same_size_pin['id'], 'role': 'same_size', 'path': str(same_size_path), 'sha256': same_size_verification['sha256']},
    {'id': bonsai_pin['id'], 'role': 'conditional', 'path': str(bonsai_path), 'sha256': bonsai_verification['sha256']},
]
execution_manifest = {
    'server_binary': str(LLAMA_SERVER),
    'server_sha256': sha256_file(LLAMA_SERVER),
    'cases_path': str(REPO / launch_settings['case_bank']),
    'output_dir': str(OUTPUT_DIR),
    'runner_code_revision': runner_revision,
    'seed': launch_settings['seed'],
    'port': 18200,
    'gpu_layers': launch_settings['gpu_layers'] if server_backend == 'cuda' else 0,
    'health_timeout_seconds': 120,
    'request_timeout_seconds': 120,
    'models': models,
}
atomic_json(MANIFEST_PATH, execution_manifest)
manifest_sha256 = sha256_file(MANIFEST_PATH)
print({'manifest': str(MANIFEST_PATH), 'sha256': manifest_sha256, 'models': [model['id'] for model in models]})


In [ ]:
report_path = OUTPUT_DIR / 'head-to-head-report.json'
run_state_path = OUTPUT_DIR / '.bonsai-advisory-run-state.json'
previous_state = {}
if run_state_path.is_file():
    try:
        previous_state = json.loads(run_state_path.read_text(encoding='utf-8'))
    except json.JSONDecodeError:
        previous_state = {}
if report_path.is_file() and previous_state.get('manifest_sha256') == manifest_sha256:
    report = json.loads(report_path.read_text(encoding='utf-8'))
else:
    if not os.environ.get('INCUBUS_BENCHMARK_SIGNING_KEY', '').strip():
        raise RuntimeError('private benchmark signing key is unavailable')
    runtime_environment = dict(os.environ)
    runtime_environment['PYTHONPATH'] = os.pathsep.join([str(REPO / 'src'), runtime_environment.get('PYTHONPATH', '')]).rstrip(os.pathsep)
    subprocess.run([sys.executable, str(REPO / 'scripts/run_head_to_head_benchmark.py'), '--manifest', str(MANIFEST_PATH)], cwd=REPO, check=True, env=runtime_environment)
    if not report_path.is_file():
        raise RuntimeError('Bonsai advisory harness completed without its report')
    report = json.loads(report_path.read_text(encoding='utf-8'))
    atomic_json(run_state_path, {'schema_version': 1, 'manifest_sha256': manifest_sha256, 'report_sha256': sha256_file(report_path), 'comparison_scope': 'advisory_only'})
print({'verdict': report.get('verdict'), 'conditional_models': report.get('conditional_models'), 'public_winner_claim': report.get('public_winner_claim')})
